# SCIP Parameter Tuning (Optuna) — Before, Principle, Application, and Effect

SCIP has numerous parameters for separating (cuts), heuristics, presolving, and branching rules, and the default settings (`SCIP_PARAMSETTING.DEFAULT`) are designed to be "well-balanced on average". However, if you want to maximize "how much the dual bound can be tightened in a fixed time" for a specific problem class, there is room to tune to settings specialized for that class. This is a **meta-optimization** independent of improvements that change the model formulation itself (such as exact linearization of integer × continuous), and SCIP itself does not do this automatically (since it is designed to work generally across problem classes).

This notebook follows this method with the following consistent pattern:

1. **Issue (before)** — Check the gap remaining even after exact linearization using `mk.analyze`
2. **Principle (principle)** — See how the gap-vs-time convergence curve changes by actually solving with 3 different combinations of separation/heuristics/branching intensities
3. **Application (how)** — Search for settings that maximize the dual bound using `minlpkit.tune.tune` (Optuna/TPE)
4. **Effect (before/after)** — Compare the default settings and the tuned settings using `mk.compare_variants`

The subject is **Batch Reactor Scheduling (Linearized)** (`samples/others/scheduling_plant.py`, `linearize_ns=True`). Even after applying [Exact Linearization of Integer × Continuous](01_linearize_product.ipynb), an unclosed gap remains, and parameter tuning acts as the "final touch after fixing the formulation" (FINDINGS §3).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.live.monitor import SolveMonitor
from minlpkit.tune import tune
from pyscipopt import SCIP_PARAMSETTING

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Issue (before) — Gap Remaining After Exact Linearization

`scheduling_plant.build_model(linearize_ns=True)` already includes the processing applied in [Exact Linearization of Integer × Continuous](01_linearize_product.ipynb) (the McCormick relaxation gap for $n\cdot s$ is already resolved).
Even so, when run through `mk.analyze`, we see that a gap still remains within a short time frame.

In [ ]:
report = mk.analyze(lambda: sp.build_model(linearize_ns=True),
                    name="plant-linearized", time_limit=8)
print(report.summary())
print("\n残存gap:", f"{report.metrics.get('gap', float('nan')):.1%}"
      if report.metrics.get("gap") is not None else "N/A")

Since the relaxation gap itself has been resolved by linearization, the remaining gap stems from search dynamics (branching order, cutting planes, heuristic hits). From here on, it's time for parameter tuning.

## 2. Principle (principle) — Search Efficiency Changes with Parameters

Even if the same model is solved for the same amount of time, "how much the dual bound tightens within a fixed time" changes depending on SCIP's settings. If separation (cut generation) is strengthened, the lower bound at the root tightens, but one round becomes heavier. If heuristics are strengthened, good upper bounds (primal solutions) are found earlier, but the time allocated for the search decreases.
Which balance is better depends on the problem class and cannot be determined theoretically.

Let's manually try out 3 settings (we won't decide which is best yet; we will let Optuna search in the next section).

In [ ]:
def apply_default(m):
    pass

def apply_config_a(m):
    m.setSeparating(SCIP_PARAMSETTING.FAST)
    m.setHeuristics(SCIP_PARAMSETTING.FAST)
    m.setParam("branching/mostinf/priority", 1000000)

def apply_config_b(m):
    m.setSeparating(SCIP_PARAMSETTING.AGGRESSIVE)
    m.setHeuristics(SCIP_PARAMSETTING.OFF)

TIME_LIMIT = 8.0

def run_with_monitor(apply_params):
    m = sp.build_model(linearize_ns=True)
    m.hideOutput()
    m.setParam("timing/clocktype", 2)
    apply_params(m)
    mon = SolveMonitor(min_interval=0.02)
    m.includeEventhdlr(mon, "monitor", "collect dual bound over time")
    m.setParam("limits/time", TIME_LIMIT)
    m.optimize()
    return mon.to_frame(), m.getDualbound()

configs = {
    "default": (apply_default, C["muted"]),
    "config A: separating=fast, heuristics=fast, branching=mostinf": (apply_config_a, C["s1"]),
    "config B: separating=aggressive, heuristics=off": (apply_config_b, C["warn"]),
}
curves = {}
for label, (fn, color) in configs.items():
    df, dual = run_with_monitor(fn)
    curves[label] = (df, color)
    print(f"{label:32s} final dual={dual:.2f}")

In [ ]:
fig = go.Figure(layout=base_layout(
    "SCIPパラメータの違いによる双対境界の収束(同一モデル・同一時間予算)",
    "求解時間 [s]", "双対境界(高いほど良い)", height=420))
for label, (df, color) in curves.items():
    d = df.dropna(subset=["dual"])
    fig.add_trace(go.Scatter(x=d["time"], y=d["dual"], mode="lines", line_shape="hv",
        line=dict(color=color, width=2.5), name=label,
        hovertemplate=label + ": t=%{x:.1f}s dual=%{y:.1f}<extra></extra>"))
show(fig)

For the execution results, refer to `final dual=` in the cell output (sometimes all 3 settings reach a higher dual bound than the default, or sometimes the opposite — you can't tell in advance which will win just by intuition). The important point is **the fact that "the destination within the same time budget changes depending on the parameter choice"**, and it is not a simple relationship of "stronger is better". Since exhaustively testing combinations by hand is inefficient, we will have Optuna do an automated search in the next section.

## 3. Application (how) — `minlpkit.tune.tune`

Instead of manually checking 3 settings, you can automatically search for combinations of separation/heuristics/presolve/branching rules using Optuna (TPE sampler). With a single line call, it also returns the percentage improvement over the default settings.

In [ ]:
res = tune(n_trials=12, time_limit=6.0)
gain = (res["best_dual"] - res["default_dual"]) / abs(res["default_dual"]) * 100
print(f"default_dual={res['default_dual']:.2f}  best_dual={res['best_dual']:.2f}  (+{gain:.1f}%)")
print("best_params:", res["best_params"])

## 4. Effect (before/after) — `mk.compare_variants`

We solve with the default settings and the best settings found by Optuna under the same time limit, and compare their **root dual bounds, final gaps, and node counts**.

In [ ]:
_EMPHASIS = {"default": SCIP_PARAMSETTING.DEFAULT, "aggressive": SCIP_PARAMSETTING.AGGRESSIVE,
             "fast": SCIP_PARAMSETTING.FAST, "off": SCIP_PARAMSETTING.OFF}

def apply_best(m, params):
    m.setSeparating(_EMPHASIS[params["separating"]])
    m.setHeuristics(_EMPHASIS[params["heuristics"]])
    m.setPresolve(_EMPHASIS[params["presolving"]] if params["presolving"] != "off"
                  else SCIP_PARAMSETTING.OFF)
    m.setParam(f"branching/{params['branching']}/priority", 1000000)

def build_default():
    return sp.build_model(linearize_ns=True)

def build_tuned():
    m = sp.build_model(linearize_ns=True)
    apply_best(m, res["best_params"])
    return m

df = mk.compare_variants({"default": build_default, "tuned": build_tuned}, time_limit=7.0)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
base = df.loc[df["variant"] == "default"].iloc[0]
tun  = df.loc[df["variant"] == "tuned"].iloc[0]

def isnan(v):
    return v is None or v != v

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%](低いほど良い。可行解0なら定義不可)",
                    "探索ノード数 (少ないほど良い)"))
labels = ["default", "tuned"]
bar_colors = [C["muted"], C["s1"]]
def add_bars(col, values, texts):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=texts, textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], tun["root_dual"]],
         [f"{base['root_dual']:.0f}", f"{tun['root_dual']:.0f}"])
gap_vals = [0.0 if isnan(base["final_gap"]) else base["final_gap"] * 100,
            100.0 if isnan(tun["final_gap"]) else tun["final_gap"] * 100]
gap_texts = ["可行解なし" if isnan(base["final_gap"]) else f"{base['final_gap']*100:.0f}%",
             "可行解なし" if isnan(tun["final_gap"]) else f"{tun['final_gap']*100:.0f}%"]
add_bars(2, gap_vals, gap_texts)
add_bars(3, [base["nodes"], tun["nodes"]], [f"{int(base['nodes'])}", f"{int(tun['nodes'])}"])
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="SCIPパラメータチューニングの before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [ ]:
def fmt_gap(v):
    return "可行解なし(gap未定義)" if isnan(v) else f"{v:.1%}"
print(f"ルート双対境界 : {base['root_dual']:.1f} -> {tun['root_dual']:.1f}")
print(f"最終dual境界   : {base['final_dual']:.1f} -> {tun['final_dual']:.1f}")
print(f"最終gap        : {fmt_gap(base['final_gap'])} -> {fmt_gap(tun['final_gap'])}")
print(f"ノード数       : {int(base['nodes'])} -> {int(tun['nodes'])}")

## Summary

- **Looking only at the dual bound, the tuned settings achieved a higher value than the default in this notebook's execution** (see the cell output above). This is the same directional effect as the preliminary measurement in FINDINGS §3 (dual bound in fixed 7 seconds: 134.8 → 143.7, +6.6%).
- **However, the best settings found by Optuna (including `heuristics=off`) failed to find even a single feasible solution within the 7-second time limit** (`final_gap` is undefined = "no feasible solution"). Because the objective function of `tune()` is purely "the dual bound in a fixed time" and does not look at primal performance at all, the settings that maximize the dual bound completely disabled heuristics, sacrificing the "solution returned within time" needed in practice.
  This is a practical example showing that **improving the dual bound and getting a usable solution are separate axes**, and blindly adopting the objective function of tuning can backfire.

### Why SCIP Doesn't Do It Automatically

SCIP's default parameters are designed to be "good on average for unknown problem classes."
The decision that "for this problem class, it is advantageous in a fixed time to weaken separation and heuristics and rely on branching" cannot be detected by presolve on a single built model; it is only known by **actually solving representative instances many times**. This is a meta-optimization on a per-problem-class basis, representing a division of labor that the modeler/operator must explicitly perform.

### When It Is Ineffective / Cautions

- **The meaning of "good settings" changes depending on the choice of the objective function**. As shown in the execution example above, settings that solely maximize the dual bound might aggressively tighten the lower bound at the expense of a primal solution. Before adopting in production, always check the primal side (final_gap/nodes/whether a solution actually returns) with `mk.compare_variants`.
- Since the search is specialized to specific instances (or representative instance groups), it does not necessarily generalize to instances of different scales or structures. In operation, it is practical to tune on representative instance groups and fix the production settings.
- If the formulation itself is weak (loose relaxations), it cannot be bridged by parameter tuning. First consider improvements on the formulation side, such as [1. Exact Linearization](../../playbook/01-linearize.md) or [2. PWL-SOS2](../../playbook/02-pwl-sos2.md).

Related: [Methods Guide 4. SCIP Parameter Tuning](../../playbook/04-tuning.md) /
API [`minlpkit.tune.tune`](../../api/tune.md) / [`mk.sweep`/`mk.rerun`](../../api/live.md).